In [2]:

# ============================================================
# ORACLE CLOUD | Cloud Future Simulator
# Complete interactive application in ONE CELL
# Run in Google Colab or Jupyter Notebook
# ============================================================

import sys, subprocess, importlib.util

packages = {
    "gradio": "gradio",
    "networkx": "networkx",
    "plotly": "plotly",
    "pandas": "pandas",
    "numpy": "numpy"
}
missing = [p for m, p in packages.items()
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

import gradio as gr
import networkx as nx
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime

# 1. CLOUD INFRASTRUCTURE
SERVICES = {
    "API Gateway": ("Gateway", 28, 8),
    "Auth Service": ("Service", 37, 8),
    "Web Frontend": ("Service", 42, 7),
    "Order Service": ("Service", 51, 9),
    "Payment Service": ("Service", 47, 10),
    "Inventory Service": ("Service", 35, 7),
    "Message Queue": ("Queue", 24, 6),
    "Customer DB": ("Database", 58, 10),
    "Analytics DB": ("Database", 32, 4),
    "Cache Cluster": ("Cache", 30, 5),
    "Notifications": ("Service", 22, 3),
    "Object Storage": ("Storage", 18, 5)
}

EDGES = [
    ("API Gateway", "Auth Service"),
    ("API Gateway", "Web Frontend"),
    ("Web Frontend", "Order Service"),
    ("Web Frontend", "Inventory Service"),
    ("Order Service", "Payment Service"),
    ("Order Service", "Inventory Service"),
    ("Order Service", "Message Queue"),
    ("Payment Service", "Customer DB"),
    ("Inventory Service", "Customer DB"),
    ("Message Queue", "Notifications"),
    ("Message Queue", "Analytics DB"),
    ("Analytics DB", "Object Storage"),
    ("Web Frontend", "Cache Cluster"),
    ("Order Service", "Cache Cluster")
]

G = nx.DiGraph()
for name, (kind, cpu, criticality) in SERVICES.items():
    G.add_node(name, kind=kind, cpu=cpu, criticality=criticality)
G.add_edges_from(EDGES)
POS = nx.spring_layout(G, seed=42, k=1.25, iterations=100)

COLORS = {
    "HEALTHY": "#38bdf8",
    "DEGRADED": "#fbbf24",
    "OVERLOADED": "#fb923c",
    "FAILED": "#ef4444",
    "FAILOVER": "#34d399"
}

# 2. SIMULATION ENGINE
def simulate(failure, traffic, threshold, failover):
    affected = (
        nx.descendants(G, failure) | {failure}
        if failure != "No failure" else set()
    )

    rows = []
    for name, (kind, base_cpu, criticality) in SERVICES.items():
        cpu = min(100, base_cpu * traffic)

        if name in affected and name != failure:
            cpu = min(100, cpu * 1.25)

        if name == failure:
            status = "FAILOVER" if failover else "FAILED"
            if not failover:
                cpu = 100
        elif cpu > threshold:
            status = "OVERLOADED"
        elif name in affected:
            status = "DEGRADED"
        else:
            status = "HEALTHY"

        rows.append({
            "Service": name,
            "Type": kind,
            "CPU (%)": round(cpu, 1),
            "Criticality": criticality,
            "Status": status
        })

    df = pd.DataFrame(rows)

    # Illustrative heuristic, not a production reliability metric
    penalties = {
        "HEALTHY": 0,
        "DEGRADED": 0.20,
        "FAILOVER": 0.10,
        "OVERLOADED": 0.65,
        "FAILED": 1.0
    }
    weights = df["Criticality"].astype(float)
    risk = df["Status"].map(penalties).astype(float)
    score = round(100 * (
        1 - (weights * risk).sum() / weights.sum()
    ), 1)

    return df, affected, score

# 3. NETWORK VISUALIZATION
def network_chart(df, failure):
    edge_x, edge_y = [], []
    for a, b in G.edges():
        x1, y1 = POS[a]
        x2, y2 = POS[b]
        edge_x += [x1, x2, None]
        edge_y += [y1, y2, None]

    edges = go.Scatter(
        x=edge_x, y=edge_y,
        mode="lines",
        line=dict(color="rgba(130,160,200,0.4)", width=1.5),
        hoverinfo="skip"
    )

    status = dict(zip(df["Service"], df["Status"]))
    xs, ys, colors, sizes, labels, hover = [], [], [], [], [], []

    for n in G.nodes():
        x, y = POS[n]
        xs.append(x)
        ys.append(y)
        colors.append(COLORS[status[n]])
        sizes.append(31 if n == failure else 23)
        labels.append(n)
        hover.append(
            f"<b>{n}</b><br>Status: {status[n]}<br>"
            f"CPU: {df.loc[df.Service == n, 'CPU (%)'].iloc[0]}%"
        )

    nodes = go.Scatter(
        x=xs, y=ys, mode="markers+text",
        text=labels, textposition="bottom center",
        textfont=dict(color="#e5eefb", size=10),
        hovertext=hover, hoverinfo="text",
        marker=dict(size=sizes, color=colors,
                    line=dict(width=1.5, color="#e5eefb"))
    )

    fig = go.Figure([edges, nodes])
    fig.update_layout(
        title="Cloud Dependency Digital Twin",
        height=570,
        paper_bgcolor="#0b1424",
        plot_bgcolor="#0b1424",
        font=dict(color="#e5eefb"),
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=50, b=10)
    )
    return fig

# 4. PERFORMANCE CHARTS
def cpu_chart(df, threshold):
    fig = go.Figure()
    for s in COLORS:
        part = df[df["Status"] == s]
        if len(part):
            fig.add_trace(go.Bar(
                x=part["Service"], y=part["CPU (%)"],
                name=s, marker_color=COLORS[s]
            ))
    fig.add_hline(
        y=threshold, line_dash="dash", line_color="#fb7185",
        annotation_text=f"Threshold {threshold}%"
    )
    fig.update_layout(
        title="Simulated CPU Utilization",
        height=400, barmode="overlay",
        yaxis=dict(title="CPU (%)", range=[0, 110]),
        xaxis=dict(tickangle=-35),
        paper_bgcolor="#0b1424", plot_bgcolor="#0b1424",
        font=dict(color="#e5eefb")
    )
    return fig

def health_chart(df):
    counts = df["Status"].value_counts()
    fig = go.Figure(go.Pie(
        labels=counts.index, values=counts.values, hole=0.62,
        marker=dict(colors=[COLORS[x] for x in counts.index]),
        textinfo="label+value"
    ))
    fig.update_layout(
        title="Service Health Distribution",
        height=330,
        paper_bgcolor="#0b1424",
        plot_bgcolor="#0b1424",
        font=dict(color="#e5eefb")
    )
    return fig

# 5. RECOMMENDATION ENGINE
def advise(df, failure, affected, traffic, failover, threshold):
    out = []
    if failure != "No failure":
        out.append(f"Investigate **{failure}**, the injected failure point.")
        descendants = sorted(nx.descendants(G, failure))
        if descendants:
            out.append(
                "Test downstream dependencies: "
                + ", ".join(descendants) + "."
            )
        out.append(
            "Validate failover health checks and data consistency."
            if failover else
            "Evaluate redundancy and test failover recovery."
        )

    overloaded = df.loc[
        df["Status"] == "OVERLOADED", "Service"
    ].tolist()
    if overloaded:
        out.append(
            "Capacity threshold exceeded for: "
            + ", ".join(overloaded) + "."
        )

    if traffic >= 2:
        out.append(
            "Run load tests and measure latency, errors, queue depth "
            "and database connection usage."
        )

    out.extend([
        "Test backup restoration and document recovery objectives.",
        "Compare several failure scenarios before changing architecture.",
        "This score is an educational heuristic, not a real outage prediction."
    ])
    return "\n\n".join(
        f"{i}. {x}" for i, x in enumerate(out, 1)
    )

# 6. DASHBOARD CALLBACK
def run(failure, traffic, threshold, failover):
    df, affected, score = simulate(
        failure, float(traffic), int(threshold), failover
    )

    affected_count = (
        len(affected) if failure != "No failure"
        else int((df.Status != "HEALTHY").sum())
    )
    bad = int(df.Status.isin(["FAILED", "OVERLOADED"]).sum())

    report = f"""
# ☁️ ORACLE CLOUD — Simulation Results

| Metric | Value |
|---|---:|
| Resilience score | **{score}/100** |
| Services modeled | **{len(df)}** |
| Blast radius / affected nodes | **{affected_count}** |
| Failed or overloaded | **{bad}** |
| Traffic multiplier | **{traffic:.2f}×** |
| Injected failure | **{failure}** |
| CPU threshold | **{threshold}%** |

Simulation time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

*Uses synthetic infrastructure data and simplified dependency rules.*
"""
    advice = advise(
        df, failure, affected, traffic, failover, threshold
    )

    csv_path = "/tmp/oracle_cloud_results.csv"
    df.to_csv(csv_path, index=False)

    return (
        report,
        network_chart(df, failure),
        cpu_chart(df, threshold),
        health_chart(df),
        df,
        advice,
        csv_path
    )

# 7. INTERACTIVE WEB APPLICATION
CSS = """
.gradio-container {
    max-width: 1500px !important;
    background: #08111f !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan", neutral_hue="slate"
    ),
    css=CSS,
    title="ORACLE CLOUD"
) as app:

    gr.Markdown("""
# ☁️ ORACLE CLOUD
### AI-Powered Cloud Future Simulator

**Simulate failures · Visualize dependencies · Explore resilience**

Synthetic cloud environment for learning and experimentation.
No real infrastructure is accessed or modified.
""")

    with gr.Row():
        failure = gr.Dropdown(
            ["No failure"] + list(SERVICES.keys()),
            value="Payment Service",
            label="Failure scenario"
        )
        traffic = gr.Slider(
            1, 5, value=1.5, step=0.25,
            label="Traffic multiplier"
        )
        threshold = gr.Slider(
            40, 95, value=75, step=5,
            label="CPU capacity threshold (%)"
        )
        failover = gr.Checkbox(
            value=True, label="Simulated failover"
        )

    button = gr.Button(
        "▶ RUN SIMULATION", variant="primary"
    )
    report = gr.Markdown()

    with gr.Tab("🌐 Cloud Architecture"):
        topology = gr.Plot()
        gr.Markdown(
            "🔵 Healthy · 🟡 Degraded · 🟠 Overloaded · "
            "🔴 Failed · 🟢 Failover"
        )

    with gr.Tab("📈 Performance Analytics"):
        cpu = gr.Plot()
        health = gr.Plot()

    with gr.Tab("📋 Service Inventory"):
        table = gr.Dataframe(interactive=False)
        download = gr.File(label="Download simulation CSV")

    with gr.Tab("🧠 Recommendations"):
        advice = gr.Markdown()

    inputs = [failure, traffic, threshold, failover]
    outputs = [report, topology, cpu, health, table, advice, download]

    button.click(run, inputs=inputs, outputs=outputs)
    app.load(run, inputs=inputs, outputs=outputs)

print("Launching ORACLE CLOUD dashboard...")
app.launch(share=True, debug=False)


/tmp/ipykernel_1523/3762077746.py:316: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Launching ORACLE CLOUD dashboard...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://010447ca40f48d421a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
